In [321]:
import os
import random
import numpy as np
import librosa

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.utils import class_weight

In [322]:
import tensorflow as tf
print(tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

2.10.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [323]:
import tensorflow as tf

In [324]:
SAMPLE_RATE = 16000
DURATION = 3
SAMPLES = SAMPLE_RATE * DURATION

SPEC_WIDTH = 94 
BATCH_SIZE = 32
EPOCHS = 10


In [325]:
BASE_DIR = r"c:\Users\RAUNAK\OneDrive\Desktop\MLDL flow\voice spoof\archive (1)"
LA_AUDIO = os.path.join(BASE_DIR, "LA", "LA", "ASVspoof2019_LA_train", "flac")
LA_LABEL = os.path.join(BASE_DIR, "LA", "LA", "ASVspoof2019_LA_cm_protocols", "ASVspoof2019.LA.cm.train.trn.txt")

MODEL_PATH = os.path.join(BASE_DIR, "models/spoof_model.h5")

In [326]:
def extract_spectrogram(file_path):
    try:
        audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)
        # Standardize length
        if len(audio) < SAMPLES:
            audio = np.pad(audio, (0, SAMPLES - len(audio)))
        else:
            audio = audio[:SAMPLES]
        
        # Convert to Mel Spectrogram
        mel = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE, n_mels=128)
        mel_db = librosa.power_to_db(mel).astype(np.float32)
        return mel_db
    except:
        return np.zeros((128, SPEC_WIDTH), dtype=np.float32)

In [327]:
def load_full_la_dataset():
    paths = []
    labels = []

    if not os.path.exists(LA_LABEL):
        print("Label file not found!")
        return [], []

    with open(LA_LABEL, "r") as f:
        for line in f:
            parts = line.strip().split()
            file_id, label = parts[1], parts[-1]
            path = os.path.join(LA_AUDIO, file_id + ".flac")
            
            if os.path.exists(path):
                paths.append(path)
                # 0 = Real, 1 = AI/Deepfake
                labels.append(0 if label == "bonafide" else 1)

    print(f"📊 Full Dataset Loaded: {len(paths)} Total Files")
    print(f"Real: {labels.count(0)} | AI: {labels.count(1)}")
    
    return paths, labels

In [328]:
def get_tf_dataset(paths, labels):
    """Efficiently feeds data to the GPU."""
    def process(path, label):
        spec = tf.numpy_function(lambda x: extract_spectrogram(x.decode()), [path], tf.float32)
        spec.set_shape((128, SPEC_WIDTH))
        return tf.expand_dims(spec, -1), label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    return ds.shuffle(2000).map(process, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [329]:


def build_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(128, SPEC_WIDTH, 1)),
        MaxPooling2D(),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(),
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        Dropout(0.5), # High dropout to prevent overfitting
        Dense(2, activation='softmax') # 2 Classes: Real and AI
    ])
    model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [330]:
model.summary()

Model: "sequential_16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_47 (Conv2D)          (None, 126, 92, 32)       320       
                                                                 
 max_pooling2d_47 (MaxPoolin  (None, 63, 46, 32)       0         
 g2D)                                                            
                                                                 
 conv2d_48 (Conv2D)          (None, 61, 44, 64)        18496     
                                                                 
 max_pooling2d_48 (MaxPoolin  (None, 30, 22, 64)       0         
 g2D)                                                            
                                                                 
 global_average_pooling2d_16  (None, 64)               0         
  (GlobalAveragePooling2D)                                       
                                                     

In [338]:
def predict_voice(file_path, model_instance):
    spec = extract_spectrogram(file_path)
    spec_input = np.expand_dims(spec, axis=(0, -1))
    
    prediction = model_instance.predict(spec_input, verbose=0)[0]
    prob_real, prob_ai = prediction[0], prediction[1]

    # --- UPDATED LOGIC ---
    # This picks whichever score is actually higher
    if prob_real > prob_ai:
        decision = "Real (Human)"
        conf = prob_real
    else:
        decision = "Deepfake (AI)"
        conf = prob_ai
        
    print(f"\nFILE: {os.path.basename(file_path)}")
    print(f"RESULT: {decision} ({conf*100:.2f}%)")
    print(f"SCORES: Real: {prob_real:.4f} | AI: {prob_ai:.4f}")

In [332]:
paths, labels = load_full_la_dataset()
    
    # B. Calculate Class Weights
weights = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=np.unique(labels),
        y=labels
    )
cw_dict = {i: weights[i] for i in range(len(weights))}

    # C. Split and Prepare
combined = list(zip(paths, labels))
random.shuffle(combined)
    
# IMPORTANT: Convert the zipped tuples back into lists
paths_shuffled, labels_shuffled = zip(*combined)
paths_shuffled = list(paths_shuffled)
labels_shuffled = list(labels_shuffled)

split = int(0.85 * len(paths_shuffled))
    
# Now passing explicit lists to avoid the Rank 0 error
train_ds = get_tf_dataset(paths_shuffled[:split], labels_shuffled[:split])
val_ds = get_tf_dataset(paths_shuffled[split:], labels_shuffled[split:])
    
# D. Train
model = build_model()
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=cw_dict)

model_name = "spoof_model.h5"
model.save(model_name)

print("-" * 30)
print(f"✅ Training finished and model saved as: {model_name}")
print("-" * 30)

📊 Full Dataset Loaded: 25380 Total Files
Real: 2580 | AI: 22800
Epoch 1/10
675/675 [==============================] - 204s 301ms/step - loss: 0.7272 - accuracy: 0.5756 - val_loss: 0.5990 - val_accuracy: 0.6047
Epoch 2/10
675/675 [==============================] - 219s 324ms/step - loss: 0.5283 - accuracy: 0.7000 - val_loss: 0.4660 - val_accuracy: 0.7730
Epoch 3/10
675/675 [==============================] - 220s 325ms/step - loss: 0.3648 - accuracy: 0.8077 - val_loss: 0.3458 - val_accuracy: 0.8290
Epoch 4/10
675/675 [==============================] - 213s 315ms/step - loss: 0.2802 - accuracy: 0.8552 - val_loss: 0.3870 - val_accuracy: 0.8411
Epoch 5/10
675/675 [==============================] - 210s 311ms/step - loss: 0.2427 - accuracy: 0.8775 - val_loss: 0.2739 - val_accuracy: 0.8752
Epoch 6/10
675/675 [==============================] - 218s 323ms/step - loss: 0.2101 - accuracy: 0.8948 - val_loss: 0.2262 - val_accuracy: 0.8926
Epoch 7/10
675/675 [==============================] - 216s 3

In [340]:

EVAL_AUDIO = os.path.join(BASE_DIR, "LA", "LA", "ASVspoof2019_LA_eval", "flac")


import glob
eval_files = glob.glob(os.path.join(EVAL_AUDIO, "*.flac"))[:50]

print("🧪 Testing Model on New Evaluation Files:")
for f in eval_files:
   
    predict_voice(f, model)

🧪 Testing Model on New Evaluation Files:

FILE: LA_E_1000147.flac
RESULT: Deepfake (AI) (99.95%)
SCORES: Real: 0.0005 | AI: 0.9995

FILE: LA_E_1000273.flac
RESULT: Deepfake (AI) (99.99%)
SCORES: Real: 0.0001 | AI: 0.9999

FILE: LA_E_1000791.flac
RESULT: Deepfake (AI) (99.99%)
SCORES: Real: 0.0001 | AI: 0.9999

FILE: LA_E_1000841.flac
RESULT: Deepfake (AI) (100.00%)
SCORES: Real: 0.0000 | AI: 1.0000

FILE: LA_E_1000989.flac
RESULT: Deepfake (AI) (99.84%)
SCORES: Real: 0.0016 | AI: 0.9984

FILE: LA_E_1001227.flac
RESULT: Deepfake (AI) (98.80%)
SCORES: Real: 0.0120 | AI: 0.9880

FILE: LA_E_1001232.flac
RESULT: Real (Human) (62.92%)
SCORES: Real: 0.6292 | AI: 0.3708

FILE: LA_E_1001320.flac
RESULT: Deepfake (AI) (97.72%)
SCORES: Real: 0.0228 | AI: 0.9772

FILE: LA_E_1001476.flac
RESULT: Deepfake (AI) (99.89%)
SCORES: Real: 0.0011 | AI: 0.9989

FILE: LA_E_1001893.flac
RESULT: Deepfake (AI) (99.88%)
SCORES: Real: 0.0012 | AI: 0.9988

FILE: LA_E_1001964.flac
RESULT: Deepfake (AI) (99.98%)
SCO